# CCI Pullback Long on SPY
## Strategy Brief
The CCI Pullback Long strategy is a mean-reversion trading strategy applied to SPY, the ETF that tracks the S&P 500 index. The strategy uses the Commodity Channel Index (CCI) to identify potential buying opportunities when the CCI indicates that the asset is oversold. A long position is taken when the CCI crosses above a specified threshold from below, suggesting a potential price rebound. The strategy aims to capitalize on short-term price corrections and is evaluated against a buy-and-hold benchmark.
## References
- https://cci-online.org/

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our CCI Pullback Long strategy. These include the CCI period, the threshold for entering a long position, and the lookback period for analysis.

In [ ]:
# Configuration
CCI_PERIOD = 20
CCI_THRESHOLD = -100
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
TICKER = 'SPY'

## PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance, compute the CCI indicator, and visualize it alongside the price data to understand its behavior.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Compute CCI
def compute_cci(data, period=CCI_PERIOD):
    TP = (data['High'] + data['Low'] + data['Close']) / 3
    CCI = (TP - TP.rolling(window=period).mean()) / (0.015 * TP.rolling(window=period).std())
    return CCI

data['CCI'] = compute_cci(data)

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.plot(data['CCI'], label='CCI', linestyle='--')
plt.axhline(CCI_THRESHOLD, color='r', linestyle='--', label='CCI Threshold')
plt.title('SPY Price and CCI Indicator')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We create a signal series based on the CCI indicator. A buy signal is generated when the CCI crosses above the threshold from below. We then define the logic for entering and exiting trades based on this signal.

In [ ]:
# Generate signals
signals = pd.Series(index=data.index, data=0)
signals[(data['CCI'] < CCI_THRESHOLD) & (data['CCI'].shift(1) >= CCI_THRESHOLD)] = 1

# Define positions
positions = signals.cumsum()

# Plot signals
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.plot(data['CCI'], label='CCI', linestyle='--')
plt.scatter(data.index, data['Close'], marker='^', color='g', label='Buy Signal', s=100, alpha=0.7, where=signals == 1)
plt.axhline(CCI_THRESHOLD, color='r', linestyle='--', label='CCI Threshold')
plt.title('SPY Price and CCI Buy Signals')
plt.legend()
plt.show()

## PHASE 4 - Coding & Backtesting
We backtest the strategy by calculating daily returns based on the positions taken. The strategy's performance is visualized through an equity curve plot.

In [ ]:
# Shift positions for backtesting
positions = positions.shift(1).fillna(0)

# Calculate daily returns
returns = data['Close'].pct_change()
strategy_returns = positions * returns

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.plot((1 + returns).cumprod(), label='Buy and Hold Equity Curve', linestyle='--')
plt.title('Equity Curve of CCI Pullback Long Strategy vs Buy and Hold')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We evaluate the strategy's performance using metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. These are compared against a buy-and-hold strategy.

In [ ]:
# Performance metrics
def calculate_performance_metrics(strategy_returns, benchmark_returns):
    # CAGR
    total_return = strategy_returns.add(1).prod() - 1
    n_years = len(strategy_returns) / 252
    cagr = (1 + total_return) ** (1 / n_years) - 1
    
    # Sharpe Ratio
    sharpe_ratio = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
    
    # Sortino Ratio
    downside_returns = strategy_returns[strategy_returns < 0]
    sortino_ratio = strategy_returns.mean() / downside_returns.std() * np.sqrt(252)
    
    # Calmar Ratio
    max_drawdown = (equity_curve.cummax() - equity_curve).max()
    calmar_ratio = cagr / max_drawdown
    
    # Max Drawdown
    max_drawdown = max_drawdown / equity_curve.cummax().max()
    
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd = calculate_performance_metrics(strategy_returns, returns)
bh_cagr, bh_sharpe, bh_sortino, bh_calmar, bh_max_dd = calculate_performance_metrics(returns, returns)

# Display results
performance_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'CCI Pullback Long': [strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd],
    'Buy and Hold': [bh_cagr, bh_sharpe, bh_sortino, bh_calmar, bh_max_dd]
})

performance_df.set_index('Metric', inplace=True)
print(performance_df)

## PHASE 6 - Deploy & Monitor
We implement a function to download the latest 60 days of SPY data, compute the CCI, and determine the current trading signal based on the strategy.

In [ ]:
def get_latest_signal(ticker=TICKER, period=CCI_PERIOD, threshold=CCI_THRESHOLD):
    # Download last 60 days
    recent_data = yf.download(ticker, period='60d')
    
    # Compute CCI
    recent_data['CCI'] = compute_cci(recent_data, period)
    
    # Determine signal
    last_cci = recent_data['CCI'].iloc[-1]
    previous_cci = recent_data['CCI'].iloc[-2]
    
    if last_cci > threshold and previous_cci <= threshold:
        print('Current Signal: BUY')
    else:
        print('Current Signal: HOLD')

# Get the latest signal
get_latest_signal()